# Gathering data on Asylum Decisions from TRAC

**Julia Ingram / CBS News**

This notebook fetches monthly asylum-case counts from the [TRAC Immigration tool](https://tracreports.org/phptools/immigration/asylum/) for one or more immigration courts, broken down by **outcome** (total / granted / denied / other) and **demographic dimension** (overall, legal representation, language, custody status).

In [1]:
import time
import os
import requests
import pandas as pd

os.makedirs("data/exported", exist_ok=True)

## Configuration

In [2]:
# Base URL
TRAC_URL = "https://tracreports.org/phptools/immigration/asylum/graph.php"

# Courts to scrape
# Key = city label | Value = TRAC's numeric city code (or 'All' for the whole country)
COURTS = {
    "Philadelphia": "30",
    "United States": "All",
}

# Outcome codes
# Key = column label | Value = TRAC's 'denial2' parameter
OUTCOMES = {
    "total":   "All",
    "granted": "2",
    "denied":  "1",
    "other":   "3",
}

# Demographic variables
# Key = Variable name | Value = another dict containing the variable's possible values
# The dimension name is the parameter sent to TRAC as part of the API request endpoint (e.g. 'represented=2').
# The key 'none' will ensure no additional filters are added to the request URL.
# This dataframe can be expanded to include additonal variables and codes that TRAC offers

OTHER_VARIABLES = {
    "none": {
        "total_cases": "All",
    },
    "represented": {
        "represented":     "2",
        "not_represented": "1",
    },
    "language": {
        "total_lang": "All",
        "english": "16",
    },
    "custody": {
        "never_detained": "1",
        "released":       "2",
        "detained":       "3",
        "not_known":      "4",
    },
}

## API request function

In [3]:
# Function takes in details of the request, builds a request URL, and returns a dataframe with the results by month
# Parameters:
#     court_code  : TRAC city ID (e.g. '30' for Philadelphia)
#     denial_code : TRAC outcome code ('All', '1'=denied, '2'=granted, '3'=other)
#     demographic_filter  : Optional URL parameter name for a demographic filter (default None)
#     demographic_value : The value paired with demographic_filter (e.g. '2' for represented) (default None).

def get_trac_data(court_code, denial_code, demographic_filter=None, demographic_value=None):
    
    params = {
        "stat":      "count",
        "timescale": "fymon",
        "base_city": court_code,
        "denial2":   denial_code,
        "timeunit":  "number",
    }

    # Add the demographic filter only when one is requested
    if demographic_filter:
        params[demographic_filter] = demographic_value

    response = requests.get(TRAC_URL, params=params, timeout=15)
    response.raise_for_status()

    timeline = response.json().get("timeline") or []
    if not timeline:
        # TRAC returns nothing when a filter has zero cases for all months
        return pd.DataFrame(columns=["fymon", "count"])

    df = pd.DataFrame(timeline)
    # 'number' can arrive as a comma-formatted string (e.g. '1,234'); clean it
    df["count"] = pd.to_numeric(
        df["number"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    return df[["fymon", "count"]]

## Fetching the data

In [4]:
# This block of code iterated through four nested loops to gather each variation of the data we need
# First we go through the different courts/cities, then by each demographic variable, then by each category 
# within that variable (e.g. represented vs not represented), and finally by each outcome type (total, granted, denied, other).
# It pulls each of these values from the dictionaries we configured up top. 

# This cell should take about 30 seconds to run

all_records = []

for court_name, court_id in COURTS.items():
    for dim_name, targets in OTHER_VARIABLES.items():
        print(f"Fetching {court_name} — {dim_name.upper()}")

        for category_label, category_value in targets.items():
            for outcome_name, denial_code in OUTCOMES.items():

                df = get_trac_data(
                    court_code=court_id,
                    denial_code=denial_code,
                    demographic_filter=dim_name,
                    demographic_value=category_value,
                )

                if not df.empty:
                    df = df.assign(
                        court=court_name,
                        dimension=dim_name,
                        category=category_label,
                        outcome=outcome_name,
                    )
                    all_records.append(df)

                time.sleep(0.3) #seconds

# Combine everything into one master table
all_data = pd.concat(all_records, ignore_index=True)
all_data.head()

Fetching Philadelphia — NONE
Fetching Philadelphia — REPRESENTED
Fetching Philadelphia — LANGUAGE
Fetching Philadelphia — CUSTODY
Fetching United States — NONE
Fetching United States — REPRESENTED
Fetching United States — LANGUAGE
Fetching United States — CUSTODY


,fymon,count,court,dimension,category,outcome
0,2000-10,68,Philadelphia,none,total_cases,total
1,2000-11,46,Philadelphia,none,total_cases,total
2,2000-12,56,Philadelphia,none,total_cases,total
3,2001-01,59,Philadelphia,none,total_cases,total
4,2001-02,42,Philadelphia,none,total_cases,total


## Reshape tables

The `all_data` table we created above is split by dimension and pivoted so each outcome becomes its own column

In [5]:
# Totals 
totals = (
    all_data
    .query("dimension == 'none'")
    .pivot_table(index=["court", "fymon"], columns="outcome", values="count", fill_value=0)
    .reset_index()
    .rename_axis(columns=None)
    [[ "court", "fymon", "total", "granted", "denied", "other" ]]
    .sort_values(["court", "fymon"])
)
totals.head()

,court,fymon,total,granted,denied,other
0,Philadelphia,2000-10,68.0,34.0,32.0,2.0
1,Philadelphia,2000-11,46.0,19.0,27.0,0.0
2,Philadelphia,2000-12,56.0,25.0,31.0,0.0
3,Philadelphia,2001-01,59.0,17.0,40.0,2.0
4,Philadelphia,2001-02,42.0,16.0,26.0,0.0


In [6]:
# Legal representation 
representation = (
    all_data
    .query("dimension == 'represented'")
    .rename(columns={"category": "representation"})
    .pivot_table(
        index=["court", "representation", "fymon"],
        columns="outcome",
        values="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(columns=None)
    [[ "court", "representation", "fymon", "total", "granted", "denied", "other" ]]
    .sort_values(["court", "representation", "fymon"])
)
representation.head()

,court,representation,fymon,total,granted,denied,other
0,Philadelphia,not_represented,2000-10,20.0,3.0,17.0,0.0
1,Philadelphia,not_represented,2000-11,22.0,7.0,15.0,0.0
2,Philadelphia,not_represented,2000-12,27.0,10.0,17.0,0.0
3,Philadelphia,not_represented,2001-01,22.0,6.0,15.0,1.0
4,Philadelphia,not_represented,2001-02,11.0,3.0,8.0,0.0


In [7]:
# Custody status 
custody = (
    all_data
    .query("dimension == 'custody'")
    .rename(columns={"category": "custody_status"})
    .pivot_table(
        index=["court", "custody_status", "fymon"],
        columns="outcome",
        values="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(columns=None)
    [[ "court", "custody_status", "fymon", "total", "granted", "denied", "other" ]]
    .sort_values(["court", "custody_status", "fymon"])
)
custody.head()

,court,custody_status,fymon,total,granted,denied,other
0,Philadelphia,detained,2000-11,1.0,0.0,1.0,0.0
1,Philadelphia,detained,2000-12,1.0,0.0,1.0,0.0
2,Philadelphia,detained,2001-01,2.0,0.0,2.0,0.0
3,Philadelphia,detained,2001-02,1.0,0.0,1.0,0.0
4,Philadelphia,detained,2001-03,2.0,0.0,2.0,0.0


In [ ]:
# Language 
# TRAC only provides an overall total and an English subset, so we derive the non-English total here

# Pivot so each category/outcome pair becomes its own column
lang_wide = (
    all_data
    .query("dimension == 'language'")
    .assign(col_name=lambda d: d["category"] + "_" + d["outcome"])
    .pivot_table(
        index=["court", "fymon"],
        columns="col_name",
        values="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(columns=None)
)

# Ensure all expected columns exist (fill with 0 if TRAC returned nothing)
expected = [
    "total_lang_total", "total_lang_granted", "total_lang_denied", "total_lang_other",
    "english_total",    "english_granted",    "english_denied",    "english_other",
]
for col in expected:
    if col not in lang_wide.columns:
        lang_wide[col] = 0

# Derive non-English counts
for outcome in ["total", "granted", "denied", "other"]:
    lang_wide[f"non_english_{outcome}"] = (
        lang_wide[f"total_lang_{outcome}"] - lang_wide[f"english_{outcome}"]
    )

# Rename total_lang_* → total_*
lang_wide = lang_wide.rename(columns={
    f"total_lang_{o}": f"total_{o}" for o in ["total", "granted", "denied", "other"]
})

# Melt back to long format so we can do a final pivot
outcome_cols = ["total", "granted", "denied", "other"]
lang_long = lang_wide.melt(
    id_vars=["court", "fymon"],
    value_vars=[
        f"{lang}_{o}"
        for lang in ["total", "english", "non_english"]
        for o in outcome_cols
    ],
    var_name="col_name",
    value_name="count",
)
# Split 'english_granted' into language='english' and outcome='granted'
lang_long[["language", "outcome"]] = lang_long["col_name"].str.rsplit("_", n=1, expand=True)

language = (
    lang_long
    .pivot_table(
        index=["court", "language", "fymon"],
        columns="outcome",
        values="count",
        fill_value=0,
    )
    .reset_index()
    .rename_axis(columns=None)
    [[ "court", "language", "fymon", "total", "granted", "denied", "other" ]]
    .sort_values(
        ["court", "language", "fymon"],
        key=lambda col: col.map({"total": 0, "english": 1, "non_english": 2}).fillna(col)
        if col.name == "language" else col
    )
)

language.head(9)

,court,language,fymon,total,granted,denied,other
606,Philadelphia,total,2000-10,68.0,34.0,32.0,2.0
607,Philadelphia,total,2000-11,46.0,19.0,27.0,0.0
608,Philadelphia,total,2000-12,56.0,25.0,31.0,0.0
609,Philadelphia,total,2001-01,59.0,17.0,40.0,2.0
610,Philadelphia,total,2001-02,42.0,16.0,26.0,0.0
611,Philadelphia,total,2001-03,39.0,6.0,32.0,1.0
612,Philadelphia,total,2001-04,56.0,22.0,34.0,0.0
613,Philadelphia,total,2001-05,55.0,14.0,40.0,1.0
614,Philadelphia,total,2001-06,42.0,11.0,31.0,0.0


## Export data

In [9]:
exports = {
    "data/exported/master_totals.csv":         totals,
    "data/exported/master_representation.csv": representation,
    "data/exported/master_language.csv":        language,
    "data/exported/master_custody.csv":         custody,
}

for path, df in exports.items():
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows to {path}")

Saved 606 rows to data/exported/master_totals.csv
Saved 1,193 rows to data/exported/master_representation.csv
Saved 1,818 rows to data/exported/master_language.csv
Saved 1,785 rows to data/exported/master_custody.csv
